In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl
import pyterrier as pt
import nltk
from spacy.lang.en import English
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import os
from pathlib import Path

nlp = English()

tokenizer = nlp.tokenizer
stemmer = PorterStemmer()

from transformers import (
    TokenClassificationPipeline,
    AutoModelForTokenClassification,
    AutoTokenizer,
)
from transformers.pipelines import AggregationStrategy


#### Complex Phrase and Term Identification

In [ ]:
if not pt.java.started():
  pt.java.init()

In [ ]:
current = os.getcwd()

In [ ]:
# Index lotte/science/dev
dataset_science = pt.get_dataset('irds:lotte/science/dev')

lotte_science_path = os.path.join(current, 'indices', 'lotte_science_dev')

if not os.path.exists(lotte_science_path):
    indexer_science = pt.IterDictIndexer(lotte_science_path, fields=['text'])
    index_ref_science = indexer_science.index(dataset_science.get_corpus_iter())
else: 
    index_ref_science = os.path.join('.', 'indices', 'lotte_science_dev')

In [ ]:
# Index lotte/lifestyle/dev
dataset_lifestyle = pt.get_dataset('irds:lotte/lifestyle/dev')

lotte_lifestyle_path = os.path.join(current, 'indices', 'lotte_lifestyle_dev')

if not os.path.exists(lotte_lifestyle_path):
    indexer_lifestyle = pt.IterDictIndexer(lotte_lifestyle_path, fields=['text'])
    index_ref_lifestyle = indexer_lifestyle.index(dataset_lifestyle.get_corpus_iter())
else:
    index_ref_lifestyle = os.path.join('.', 'indices', 'lotte_lifestyle_dev')

In [ ]:
science_index = pt.IndexFactory.of(str(Path(index_ref_science).resolve()))
lifestyle_index = pt.IndexFactory.of(str(Path(index_ref_lifestyle).resolve()))

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Define keyphrase extraction pipeline, original by IRGC@SimpleText'23, code: https://colab.research.google.com/drive/10LyozPzxUlqFxHkXyfjxezO469c1ou9z?usp=sharing
class KeyphraseExtractionPipeline(TokenClassificationPipeline):
    def __init__(self, model, *args, **kwargs):
        super().__init__(
            model=AutoModelForTokenClassification.from_pretrained(model),
            tokenizer=AutoTokenizer.from_pretrained(model),
            *args,
            **kwargs
        )

    def postprocess(self, model_outputs):
        results = super().postprocess(
            all_outputs=model_outputs,
            aggregation_strategy=AggregationStrategy.SIMPLE,
        )
        return np.unique([result.get("word").strip() for result in results])

model_name = "ml6team/keyphrase-extraction-kbir-inspec"
extractor = KeyphraseExtractionPipeline(model=model_name)

def get_idf_for_term(term, index):
  term = str(term)
  lex = index.getLexicon()
  stemmed_term = stemmer.stem(term)

  df_term = lex[stemmed_term].getDocumentFrequency() if stemmed_term in lex else 1
  N = index.getCollectionStatistics().numberOfDocuments

  return np.emath.logn(N, N/df_term)

def get_idf_diffs(tokens, science_index, lifestyle_index):
  science_idfs = [get_idf_for_term(token, science_index) for token in tokens]
  lifestyle_idfs = [get_idf_for_term(token, lifestyle_index) for token in tokens]

  diffs = []
  for i in range(len(tokens)):
    science_idf = get_idf_for_term(tokens[i], science_index)
    lifestyle_idf = get_idf_for_term(tokens[i], lifestyle_index)

    if science_idf < lifestyle_idf:
      diffs.append(np.round(lifestyle_idf - science_idf, 4))

    elif science_idf == 1 and lifestyle_idf == 1:
      diffs.append(1.0)
    else:
      diffs.append(0.0)

  return diffs

def preprocess(text):
  stop_words = set(stopwords.words('english'))
  text = text.translate(str.maketrans('', '', string.punctuation))
  tokens = word_tokenize(text)
  filtered_tokens = np.array([token for token in tokens if str.lower(token) not in stop_words])

  return filtered_tokens

def identify_phrase(text):
  key_phrases = extractor(text)
  for a in key_phrases:
    text = text.replace(a, f"[{a}]")
  return text

def identify_scientific_phrase(text, complexity_threshold=0.1, method='max', verbose=False):
  phrases = extractor(text)
  supported_text = text
  diff_vals = [get_idf_diffs(preprocess(phrase), science_index, lifestyle_index) for phrase in phrases]
  diff_vals = [[0] if dv == [] else dv for dv in diff_vals]
  if len(diff_vals) == 0:
    return supported_text
  sum_vals = [sum(diff) for diff in diff_vals]
  avg_vals = [np.average(diff) for diff in diff_vals]
  max_vals = [max(diff) for diff in diff_vals]

  for i in range(len(phrases)):
    if method == 'max' and max_vals[i] >= complexity_threshold:
      supported_text = supported_text.replace(phrases[i], f"[{phrases[i]}]")
    elif method == 'avg' and avg_vals[i] >= complexity_threshold:
      supported_text = supported_text.replace(phrases[i], f"[{phrases[i]}]")
    elif method == 'sum' and sum_vals[i] >= complexity_threshold:
      supported_text = supported_text.replace(phrases[i], f"[{phrases[i]}]")

    if verbose:
      print(f"{phrases[i]} {diff_vals[i]} sum: {sum_vals[i]} avg: {avg_vals[i]} max: {max_vals[i]}")
  return supported_text

In [ ]:
# Load pipeline
model_name = "ml6team/keyphrase-extraction-kbir-inspec"
extractor = KeyphraseExtractionPipeline(model = model_name)

#### Identify complex scientific phrases in SimpleText'25 corpus

In [ ]:
df_final = pd.read_json(os.path.join('data', 'simpletext25_task11_text.json'))
prep_snt = {}

for i, r in df_final.iterrows():
    source_snt = r['complex']
    prep_snt[i] = identify_scientific_phrase(source_snt)

with open('prepared_sentences.pkl', 'wb') as f:
    pkl.dump(prep_snt, f)